### FGLS Algorithm

Following the given instructions and steps, the follwoing code was implemented by me, without any help of AI:

In [ ]:
import numpy as np

class FGLS():
    """
    The purpose of this script is to have a class that can perform the FGLS algorithm for a given dataset.
    Generally, we would want to take an input like X (the predictor variable) and y (the target variable). 
    > For simplicity of this script, this algorithm is intended to be used on a single-variable case-study.

    The suggested workflow for the FGLS algorithm is the following:
    1. Estimate beta_hat using OLS for time-stamp 0.
    2. For time-stamps $t = 0$ trought a max_iteration, perform:
        a. $e^{(t)}_i = (y_i - x^{T}_i \hat{beta}^{(t)})²$, for $i = 1, ..., n$

        b. $\hat{\gamma}^{(t)} = (X^T X)^{-1} X^T e^{(t)}$

        c. $w_i^{(t)} = \fracc{1}{X_i^T \hat{\gamma}^{(t)}}$

        d. $W^T = \diag(w_1^{(t)}, ..., w_n^{(t)})$

        e. $\hat{\beta}^{(t+1)} = (X^T W^{(t)} X)^{-1} X^T W^{(t)} y$

        f. if $||\hat{\Beta}^{(t+1)} - \hat{\Beta}^{(t)}||_p < tol$, return Beta, else go back to step a. 
    """

    def __init__(self):
        
        self.X = None   
        self.y = None
        self.error = None
        self.Omega = None
        self.beta_t = None
        self.y_hat = None


    # Step 1:
    def beta_0(self, X, y):
        ols = OLS()
        beta_hat, _ = ols.fit(X, y)
        self.beta_t = beta_hat

        return True
    
    # (Loop):
    def FGLS(self, X, y, Beta_0, max_iteration: int = 10000, alpha: float = 0.05):
        n = 1
        error = 1
        B = Beta_0

        def w_i(x_i, gamma):
            return 1/(x_i * gamma)
        
        def error(beta_0, beta_t):
            return np.linalg.norm(x = np.array(beta_0 - beta_t), ord = 2) # We asuming a single variable algorithm. 
        
        while n < max_iteration or error > alpha:

            # Update beta_t 
            self.beta_t = B

            # Step a:
            e_t = np.array([(y[i] - X[i]*B)**2] for i in range(len(X)))

            # Step b:
            gamma_t = np.linalg.solve(X.T @ X, X.T @ e_t)

            # Step c and d:
            W_t = np.diag(np.array([w_i(X[i], gamma_t) for i in range(len(X))]))

            # Step e:
            B = np.linalg.solve(X.T @ W_t @ X, X.T @ W_t @ y)

            error = error(self.beta_t, B)
            n += 1
        self.beta_t = B

        return self.beta_t


    def predict(self, X):
        X = np.array(X)
        X_aug = np.column_stack((np.ones(X.shape[0]), X))
        return X_aug @ self.beta_t
 

Nevertheless, I validated for **mathematical accuracy, computational efficiency and code quality** with an AI, and delivered a much neatter version. 

Code as follows:

In [ ]:
class FGLS:
    """
    The purpose of this class is to have a class that can perform the FGLS algorithm for a given dataset.
    Generally, we would want to take an input like X (the predictor variable) and y (the target variable). 

    The suggested workflow for the FGLS algorithm is the following:

    1. Estimate beta_hat using OLS for time-stamp 0.
    2. For time-stamps $t = 0$ trought a max_iteration, perform:

        a. $e^{(t)}_i = (y_i - x^{T}_i \hat{beta}^{(t)})²$, for $i = 1, ..., n$

        b. $\hat{\gamma}^{(t)} = (X^T X)^{-1} X^T e^{(t)}$

        c. $w_i^{(t)} = \fracc{1}{X_i^T \hat{\gamma}^{(t)}}$

        d. $W^T = \diag(w_1^{(t)}, ..., w_n^{(t)})$

        e. $\hat{\beta}^{(t+1)} = (X^T W^{(t)} X)^{-1} X^T W^{(t)} y$

        f. if $||\hat{\Beta}^{(t+1)} - \hat{\Beta}^{(t)}||_p < tol$, return Beta, else go back to step a.

    """

    def __init__(self):
        self.beta_t = None
        self.converged = False

    def fit(self, X, y, max_iteration: int = 1000, tol: float = 1e-5):
        X = np.asarray(X)
        y = np.asarray(y)
        
        if X.ndim == 1:
            X = X.reshape(-1, 1)
            
        X_aug = np.column_stack((np.ones(X.shape[0]), X))
        
        # Step 1: Getting Beta_0 with OLS
        beta_t = np.linalg.solve(X_aug.T @ X_aug, X_aug.T @ y)

        for i in range(max_iteration):
            # Step a: Vectorized squared residuals
            e_t = (y - X_aug @ beta_t)**2

            # Step b: Gamma to estimate variance
            gamma_t = np.linalg.solve(X_aug.T @ X_aug, X_aug.T @ e_t)

            # Step c: Calculate weights, were we punish high variance
            var_estimated = X_aug @ gamma_t
            
            # Safety mechanism: Variance cannot be negative or zero
            var_estimated = np.clip(var_estimated, a_min=1e-8, a_max=None)
            w = 1.0 / var_estimated

            # Step d & e: WLS using broadcasting (O(N) memory instead of O(N^2))
            # Mathematically equivalent to X^T @ W @ X, but drastically faster
            X_W = X_aug * w[:, np.newaxis] 
            
            # B_new = (X^T W X)^-1 (X^T W y)
            B_new = np.linalg.solve(X_W.T @ X_aug, X_W.T @ y)

            # Step f: Check convergence using the L2 norm
            error_diff = np.linalg.norm(B_new - beta_t, ord=2)
            
            beta_t = B_new # Update for next loop
            
            if error_diff < tol:
                self.converged = True
                print(f"Algorithm converged on {i}th iteration.")
                break
                
        self.beta_t = beta_t
        return self.beta_t

    def predict(self, X):
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        X_aug = np.column_stack((np.ones(X.shape[0]), X))
        return X_aug @ self.beta_t


Finally, as is visible on the source code, we made a function to test different parameters for the case-study, here are the results:

In [6]:
from OLS import case_1
output = case_1(n = 500, b_0 = 5, b_1 = 2, b_2 = 0.4, b_3 = -0.2)


Algorithm converged on 2th iteration.
--- Algorithm Comparison ---
True Betas : Intercept = 5.0000, Slope = 2.0000
OLS Betas  : Intercept = 4.9552, Slope = 2.0006
FGLS Betas : Intercept = 5.0007, Slope = 2.0000

--- Absolute Errors (Lower is better) ---
OLS Error  : Intercept = 0.0448, Slope = 0.0006
FGLS Error : Intercept = 0.0007, Slope = 0.0000
